**Problem 1 - Bot-aware sessionization + week-over-week retention**

In [1]:
import sys
print(sys.executable)

C:\Users\flood\anaconda3\envs\ml_env\python.exe


In [2]:
import os
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("HW2")
    .master("local[*]")
    .getOrCreate()
)

spark

In [3]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType
import re

In [4]:
event = spark.read.json("data/p1_raw_events.jsonl")
event = event.withColumn(
    "event_ts",
    F.to_timestamp("event_ts")
).withColumn(
    "ingested_at",
    F.to_timestamp("ingested_at")
)
event.printSchema()
event.show(5, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- page: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- user_id: string (nullable = true)

+--------+----------+-------------------+-------------------+----------------------+------------------------------------------------------------+-------+
|event_id|event_name|event_ts           |ingested_at        |page                  |user_agent                                                  |user_id|
+--------+----------+-------------------+-------------------+----------------------+------------------------------------------------------------+-------+
|E200000 |page_view |2026-02-24 19:06:43|2026-02-24 19:10:41|/home                 |Mozilla/5.0 (X11; Linux x86_64) Gecko/20100101 Firefox/122.0|U00000 |
|E200001 |click     |2026-02-24 19:11:11|2026-02-24 19:13:54|/help                 |Mozill

In [5]:
@F.udf(StringType())
def classify(user_agent):

    if user_agent is None:
        return "OTHER"

    s = str(user_agent).strip()

    if s == "":
        return "OTHER"

    s_lower = s.lower()

    # words that should cancel bot detection
    if any(x in s_lower for x in ["robotics", "reboot", "botanist"]):
        pass
    else:
        if re.search(r"\bbot\b", s_lower):
            return "BOT"
        if re.search(r"\bcrawler\b", s_lower):
            return "BOT"
        if re.search(r"\bspider\b", s_lower):
            return "BOT"
        if re.search(r"\bslurp\b", s_lower):
            return "BOT"
        if s_lower.startswith("curl/"):
            return "BOT"
        if s_lower.startswith("wget/"):
            return "BOT"
        if s_lower.startswith("python-requests/"):
            return "BOT"
        if "headlesschrome" in s_lower:
            return "BOT"

    if "ipad" in s_lower:
        return "TABLET"

    if "android" in s_lower and "mobile" not in s_lower:
        return "TABLET"

    if "iphone" in s_lower:
        return "MOBILE"

    if "android" in s_lower and "mobile" in s_lower:
        return "MOBILE"

    if any(x in s_lower for x in ["windows nt", "macintosh", "x11", "linux"]):
        return "DESKTOP"

    return "OTHER"


    

In [6]:
dedup_w = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("ingested_at").desc_nulls_last(),
        F.col("event_ts").desc_nulls_last(),
        F.col("event_name").asc_nulls_last(),
        F.col("page").asc_nulls_last()
    )
)

event_dedup = (
    event
    .withColumn("rn_dedup", F.row_number().over(dedup_w))
    .filter(F.col("rn_dedup") == 1)
    .drop("rn_dedup")
)

In [7]:
events_with_ua = (
    event_dedup
    .withColumn("ua_type", classify(F.col("user_agent")))
)

events_nonbot = events_with_ua.filter(F.col("ua_type") != "BOT")

events_local = (
    events_nonbot
    .withColumn("local_ts", F.from_utc_timestamp(F.col("event_ts"), "America/New_York"))
    .withColumn("local_date", F.to_date("local_ts"))
)

In [8]:
session_order_w = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.col("event_ts").asc_nulls_last(),
        F.col("ingested_at").asc_nulls_last(),
        F.col("event_id").asc_nulls_last(),
        F.col("event_name").asc_nulls_last(),
        F.col("page").asc_nulls_last()
    )
)

In [9]:
events_sessions = (
    events_local
    .withColumn("prev_event_ts", F.lag("event_ts").over(session_order_w))
    .withColumn("prev_local_date", F.lag("local_date").over(session_order_w))
    .withColumn(
        "gap_minutes",
        (
            F.unix_timestamp("event_ts") - 
        F.unix_timestamp("prev_event_ts")
        ) / 60
    )
    .withColumn(
        "is_session_start",
        F.when(F.col("prev_event_ts").isNull(), F.lit(1))
         .when(F.col("gap_minutes") > F.lit(25), F.lit(1))
         .when(F.col("local_date") != F.col("prev_local_date"), F.lit(1))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "session_index",
        F.sum("is_session_start").over(
            session_order_w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
    .withColumn(
        "session_id",
        F.concat(
            F.col("user_id"),
            F.lit("#"),
            F.lpad(F.col("session_index").cast("string"), 6, "0")
        )
    )
    .drop("prev_local_date")
)

In [10]:
events_sessions = (
    events_sessions
    .withColumn(
        "week_start_date",
        F.date_sub(F.next_day(F.to_date("local_ts"), "Mon"),7)
    )
)

In [11]:
weekly_base = (
    events_sessions
    .groupBy("user_id", "week_start_date")
    .agg(
        F.countDistinct("session_id").alias("sessions"),
        F.count(F.lit(1)).alias("events"),
        F.countDistinct(F.to_date("local_ts")).alias("active_days_in_week")
    )
)

In [12]:
first_ua_w = (
    Window
    .partitionBy("user_id", "week_start_date")
    .orderBy(
        F.col("event_ts").asc_nulls_last(),
        F.col("event_id").asc_nulls_last()
    )
)

weekly_first_ua = (
    events_sessions
    .withColumn("rn_first_week", F.row_number().over(first_ua_w))
    .filter(F.col("rn_first_week") == 1)
    .select(
        "user_id",
        "week_start_date",
        F.col("ua_type").alias("first_ua_type_in_week")
    )
)

In [13]:
retention_w = (
    Window
    .partitionBy("user_id")
    .orderBy("week_start_date")
)

weekly_retention = (
    weekly_base
    .withColumn("next_week_start", F.lead("week_start_date").over(retention_w))
    .withColumn(
        "returning_next_week",
        F.when(
            F.datediff(F.col("next_week_start"), F.col("week_start_date")) == 7,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .drop("next_week_start")
)

In [14]:
user_week_metrics = (
    weekly_retention
    .join(
        weekly_first_ua,
        on=["user_id", "week_start_date"],
        how="left"
    )
    .select(
        "user_id",
        "week_start_date",
        "sessions",
        "events",
        "first_ua_type_in_week",
        "active_days_in_week",
        "returning_next_week"
    )
)

In [21]:
events_sessions.orderBy(
    "user_id",
    "event_ts",
    "ingested_at",
    "event_id"
).show(20, truncate=False)

+--------+----------+-------------------+-------------------+---------------+---------------------------------------------------------------------------------------------+-------+-------+-------------------+----------+-------------------+-------------------+----------------+-------------+-------------+---------------+
|event_id|event_name|event_ts           |ingested_at        |page           |user_agent                                                                                   |user_id|ua_type|local_ts           |local_date|prev_event_ts      |gap_minutes        |is_session_start|session_index|session_id   |week_start_date|
+--------+----------+-------------------+-------------------+---------------+---------------------------------------------------------------------------------------------+-------+-------+-------------------+----------+-------------------+-------------------+----------------+-------------+-------------+---------------+
|E200004 |click     |2026-02-24 11:42:17

In [26]:
user_week_metrics.orderBy(
    "user_id",
    "week_start_date"
).show(20, truncate=False)

+-------+---------------+--------+------+---------------------+-------------------+-------------------+
|user_id|week_start_date|sessions|events|first_ua_type_in_week|active_days_in_week|returning_next_week|
+-------+---------------+--------+------+---------------------+-------------------+-------------------+
|U00000 |2026-02-23     |2       |8     |DESKTOP              |1                  |0                  |
|U00001 |2026-02-16     |1       |4     |MOBILE               |1                  |0                  |
|U00002 |2026-02-09     |1       |3     |MOBILE               |1                  |1                  |
|U00002 |2026-02-16     |1       |5     |DESKTOP              |1                  |1                  |
|U00002 |2026-02-23     |1       |2     |DESKTOP              |1                  |0                  |
|U00003 |2026-02-16     |1       |5     |DESKTOP              |1                  |0                  |
|U00004 |2026-02-02     |2       |6     |DESKTOP              |2

In [23]:
events_sessions.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- page: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ua_type: string (nullable = true)
 |-- local_ts: timestamp (nullable = true)
 |-- local_date: date (nullable = true)
 |-- prev_event_ts: timestamp (nullable = true)
 |-- gap_minutes: double (nullable = true)
 |-- is_session_start: integer (nullable = false)
 |-- session_index: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- week_start_date: date (nullable = true)



In [24]:
user_week_metrics.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- week_start_date: date (nullable = true)
 |-- sessions: long (nullable = false)
 |-- events: long (nullable = false)
 |-- first_ua_type_in_week: string (nullable = true)
 |-- active_days_in_week: long (nullable = false)
 |-- returning_next_week: integer (nullable = false)



In [ ]:
!